# [16.7] Data Shapley in One Training Run - Solutions

Reference validation notebook for exact, Monte Carlo, in-run, negative-control, and runtime-overhead Data Shapley checks.

In [ ]:
import json
import sys
from pathlib import Path

chapter = "chapter16_shapley_attribution_baselines"
section = "part7_data_shapley_in_one_training_run"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section

if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part7_data_shapley_in_one_training_run.tests as tests
from chapter16_shapley_attribution_baselines.exercises.part7_data_shapley_in_one_training_run import solutions

In [ ]:
tests.test_exact_data_shapley_smoke_test(solutions.exact_data_shapley_smoke_test)
tests.test_monte_carlo_data_shapley_smoke_test(solutions.monte_carlo_data_shapley_smoke_test)
tests.test_in_run_data_shapley_smoke_test(solutions.in_run_data_shapley_smoke_test)
tests.test_random_data_attribution_failure_smoke_test(solutions.random_data_attribution_failure_smoke_test)
tests.test_label_shuffled_attribution_failure_smoke_test(solutions.label_shuffled_attribution_failure_smoke_test)
tests.test_runtime_overhead_smoke_test(solutions.runtime_overhead_smoke_test)
tests.test_notebook_contract(solutions.run_smoke_test)

In [ ]:
contract = solutions.run_smoke_test(cpu=True)
assert contract["exact"]["harmful_index"] == 3
assert contract["exact"]["deletion_test_passes"]
assert contract["monte_carlo"]["approximates_exact"]
assert contract["in_run"]["correlates_with_exact"]
assert contract["random_data_control"]["random_data_attribution_fails"]
assert contract["label_shuffle_control"]["label_shuffled_attribution_fails"]
assert contract["runtime_overhead"]["runtime_overhead_reported"]
contract

In [ ]:
report = json.loads((section_dir / "verification_report.json").read_text())
gpu = report["metrics"]["gpu_test"]
assert report["accepted"]
assert gpu["cuda_available"]
assert gpu["preflight_passed"]
assert gpu["training_example_count"] == 4
assert gpu["coalition_count"] == 16
assert gpu["monte_carlo_samples"] == 512
assert gpu["harmful_index"] == 3
assert gpu["harmful_value"] < 0.0
assert gpu["harmful_removal_delta"] > 0.0
assert gpu["sampled_approximates_exact"]
assert gpu["sampled_max_abs_error"] <= 0.08
assert gpu["pearson_correlation"] >= 0.99
assert gpu["identifies_harmful"]
assert gpu["random_data_attribution_fails"]
assert gpu["random_data_max_abs_signal_correlation"] <= 0.25
assert gpu["label_shuffled_attribution_fails"]
assert gpu["label_shuffled_signal_correlation"] <= 0.0
assert gpu["runtime_overhead_reported"]
assert gpu["runtime_measurement_repeats"] == 128
assert gpu["runtime_in_run_vs_exact_ratio"] <= 2.0
assert abs(gpu["actual_full_batch_one_step_utility"] - gpu["coalition_full_utility"]) <= 1e-9
assert gpu["within_vram_budget"]
{key: gpu[key] for key in [
    "device",
    "harmful_value",
    "sampled_max_abs_error",
    "pearson_correlation",
    "random_data_max_abs_signal_correlation",
    "label_shuffled_signal_correlation",
    "runtime_in_run_vs_exact_ratio",
    "peak_vram_gb",
]}